# Mock Exam 2 — Practical Solutions


## Exercise 1 — Selenium scraping (conceptual solution)
2 ways of doing it:
* using selenium from start until end
* using selenium driver.page_source option to render html and use beautiful soup after

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd

BASE_URL = "https://books.toscrape.com/"

driver = webdriver.Chrome()
driver.get(BASE_URL)

driver.get(BASE_URL)

wait = WebDriverWait(driver, 15)
wait.until(
    EC.presence_of_all_elements_located(
        (By.CSS_SELECTOR, "article.product_pod")
    )
)

# Find all book containers
pods = driver.find_elements(By.CSS_SELECTOR, "article.product_pod")

rows = []

for pod in pods:
    # Title (stored in the title attribute of the <a> tag)
    a = pod.find_element(By.CSS_SELECTOR, "h3 a")
    title = a.get_attribute("title").strip()

    # Price
    price = pod.find_element(By.CLASS_NAME, "price_color").text.strip()

    # Availability
    availability = pod.find_element(By.CLASS_NAME, "availability").text.strip()

    rows.append({
        "title": title,
        "price_raw": price,
        "availability": availability
    })

df = pd.DataFrame(rows)

df.head()

,title,price_raw,availability
0,A Light in the Attic,£51.77,In stock
1,Tipping the Velvet,£53.74,In stock
2,Soumission,£50.10,In stock
3,Sharp Objects,£47.82,In stock
4,Sapiens: A Brief History of Humankind,£54.23,In stock


In [2]:
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"
PAGE_URL = urljoin(BASE_URL, "catalogue/page-1.html")

driver = webdriver.Chrome()
driver.get(PAGE_URL)

wait = WebDriverWait(driver, 15)
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "article.product_pod")))

html = driver.page_source
soup = BeautifulSoup(html, "html.parser")
len(html)
rows = []
for pod in soup.select("article.product_pod"):
    a = pod.select_one("h3 a")
    title = a.get("title", "").strip()
    price = pod.select_one(".price_color").get_text(strip=True)
    availability = pod.select_one(".availability").get_text(" ", strip=True)
    rows.append({"title": title, "price_raw": price, "availability": availability})

df = pd.DataFrame(rows)
df.head()

,title,price_raw,availability
0,A Light in the Attic,£51.77,In stock
1,Tipping the Velvet,£53.74,In stock
2,Soumission,£50.10,In stock
3,Sharp Objects,£47.82,In stock
4,Sapiens: A Brief History of Humankind,£54.23,In stock


## Exercise 2 — Regex extraction
Only for the purpose of the exercise, cents values from price text (£55.21 --> .21)


In [8]:
df['cents'] = df['price_raw'].str.extract(r'(\.\d{2})')
df

,title,price_raw,availability,stock_count,cents
0,A Light in the Attic,£51.77,In stock,.77,.77
1,Tipping the Velvet,£53.74,In stock,.74,.74
2,Soumission,£50.10,In stock,.10,.10
3,Sharp Objects,£47.82,In stock,.82,.82
4,Sapiens: A Brief History of Humankind,£54.23,In stock,.23,.23
5,The Requiem Red,£22.65,In stock,.65,.65
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,In stock,.34,.34
7,The Coming Woman: A Novel Based on the Life of...,£17.93,In stock,.93,.93
8,The Boys in the Boat: Nine Americans and Their...,£22.60,In stock,.60,.60
9,The Black Maria,£52.15,In stock,.15,.15


## Exercise 3 — Business join and revenue computation
Merge sales and book data and compute revenue.


In [10]:
import pandas as pd

# Sales dataset
df_sales = pd.DataFrame({
    "book_id": [1, 2, 3, 4, 6],
    "quantity_sold": [120, 85, 60, 40, 10]
})

# Books dataset
df_books = pd.DataFrame({
    "book_id": [1, 2, 3, 4, 5],
    "title": [
        "Python Basics",
        "Advanced Pandas",
        "Web Scraping 101",
        "Data Visualization",
        "Machine Learning Intro"
    ],
    "price": [25.0, 35.0, 30.0, 28.0, 45.0]
})

df_sales, df_books


(   book_id  quantity_sold
 0        1            120
 1        2             85
 2        3             60
 3        4             40
 4        6             10,
    book_id                   title  price
 0        1           Python Basics   25.0
 1        2         Advanced Pandas   35.0
 2        3        Web Scraping 101   30.0
 3        4      Data Visualization   28.0
 4        5  Machine Learning Intro   45.0)

In [12]:
df_agg = df_sales.merge(df_books, on="book_id", how="outer")

In [13]:
df_agg["revenue"] = df_agg["price"]*df_agg["quantity_sold"]

In [14]:
df_agg.head()

,book_id,quantity_sold,title,price,revenue
0,1,120.0,Python Basics,25.0,3000.0
1,2,85.0,Advanced Pandas,35.0,2975.0
2,3,60.0,Web Scraping 101,30.0,1800.0
3,4,40.0,Data Visualization,28.0,1120.0
4,5,NaN,Machine Learning Intro,45.0,NaN
